# EBAC - Regressão II - regressão múltipla

## Tarefa I

#### Previsão de renda II

Vamos continuar trabalhando com a base 'previsao_de_renda.csv', que é a base do seu próximo projeto. Vamos usar os recursos que vimos até aqui nesta base.

|variavel|descrição|
|-|-|
|data_ref                | Data de referência de coleta das variáveis |
|index                   | Código de identificação do cliente|
|sexo                    | Sexo do cliente|
|posse_de_veiculo        | Indica se o cliente possui veículo|
|posse_de_imovel         | Indica se o cliente possui imóvel|
|qtd_filhos              | Quantidade de filhos do cliente|
|tipo_renda              | Tipo de renda do cliente|
|educacao                | Grau de instrução do cliente|
|estado_civil            | Estado civil do cliente|
|tipo_residencia         | Tipo de residência do cliente (própria, alugada etc)|
|idade                   | Idade do cliente|
|tempo_emprego           | Tempo no emprego atual|
|qt_pessoas_residencia   | Quantidade de pessoas que moram na residência|
|renda                   | Renda em reais|

In [1]:
import pandas as pd

In [7]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_csv('previsao_de_renda.csv')

df['log_renda'] = np.log(df['renda'])

modelo = smf.ols(
    'log_renda ~ idade + tempo_emprego + qtd_filhos + '
    'C(sexo) + C(posse_de_veiculo) + C(posse_de_imovel) + '
    'C(tipo_renda) + C(educacao) + C(estado_civil) + C(tipo_residencia)',
    data=df
).fit()

print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:              log_renda   R-squared:                       0.357
Model:                            OLS   Adj. R-squared:                  0.356
Method:                 Least Squares   F-statistic:                     299.5
Date:                Tue, 05 May 2026   Prob (F-statistic):               0.00
Time:                        20:21:22   Log-Likelihood:                -13571.
No. Observations:               12427   AIC:                         2.719e+04
Df Residuals:                   12403   BIC:                         2.737e+04
Df Model:                          23                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             15000 non-null  int64  
 1   data_ref               15000 non-null  object 
 2   id_cliente             15000 non-null  int64  
 3   sexo                   15000 non-null  object 
 4   posse_de_veiculo       15000 non-null  bool   
 5   posse_de_imovel        15000 non-null  bool   
 6   qtd_filhos             15000 non-null  int64  
 7   tipo_renda             15000 non-null  object 
 8   educacao               15000 non-null  object 
 9   estado_civil           15000 non-null  object 
 10  tipo_residencia        15000 non-null  object 
 11  idade                  15000 non-null  int64  
 12  tempo_emprego          12427 non-null  float64
 13  qt_pessoas_residencia  15000 non-null  float64
 14  renda                  15000 non-null  float64
 15  lo

1. Separe a base em treinamento e teste (25% para teste, 75% para treinamento).
2. Rode uma regularização *ridge* com alpha = [0, 0.001, 0.005, 0.01, 0.05, 0.1] e avalie o $R^2$ na base de testes. Qual o melhor modelo?
3. Faça o mesmo que no passo 2, com uma regressão *LASSO*. Qual método chega a um melhor resultado?
4. Rode um modelo *stepwise*. Avalie o $R^2$ na vase de testes. Qual o melhor resultado?
5. Compare os parâmetros e avalie eventuais diferenças. Qual modelo você acha o melhor de todos?
6. Partindo dos modelos que você ajustou, tente melhorar o $R^2$ na base de testes. Use a criatividade, veja se consegue inserir alguma transformação ou combinação de variáveis.
7. Ajuste uma árvore de regressão e veja se consegue um $R^2$ melhor com ela.

In [39]:
# 1) Separar treino e teste (75/25)
from sklearn.model_selection import train_test_split

# remover nulos se necessário
df = df.dropna()

X = df.drop(columns=['renda', 'log_renda'])
y = df['log_renda']

# dummies
X = pd.get_dummies(X, drop_first=True)

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)


In [43]:
# 2) Ridge Regression
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

alphas = [0, 0.001, 0.005, 0.01, 0.05, 0.1]

resultados_ridge = {}

for a in alphas:
    model = Ridge(alpha=a)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    resultados_ridge[a] = r2

print(resultados_ridge)

{0: 0.365291075608139, 0.001: 0.36529155195979357, 0.005: 0.36529344586790824, 0.01: 0.3652957877817673, 0.05: 0.3653135770235417, 0.1: 0.3653337538572967}


In [47]:
# 3) LASSO
from sklearn.linear_model import Lasso

resultados_lasso = {}

for a in alphas:
    model = Lasso(alpha=a, max_iter=10000)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    resultados_lasso[a] = r2

print(resultados_lasso)

C:\Users\olive\anaconda3\Lib\site-packages\sklearn\base.py:1473: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
C:\Users\olive\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:697: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_descent(
C:\Users\olive\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.411e+03, tolerance: 7.522e-01 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinat

{0: 0.36529107560850416, 0.001: 0.36728302680983693, 0.005: 0.3678101981348987, 0.01: 0.36544232087242867, 0.05: 0.33806892849601244, 0.1: 0.2919080925407034}


In [49]:
# 4) Stepwise (manual simplificado)
# garantir que tudo é numérico
X_train = X_train.astype(float)
X_test = X_test.astype(float)

In [51]:
import statsmodels.api as sm

X_sm = sm.add_constant(X_train)

modelo = sm.OLS(y_train, X_sm).fit()

print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:              log_renda   R-squared:                       0.359
Model:                            OLS   Adj. R-squared:                  0.356
Method:                 Least Squares   F-statistic:                     123.7
Date:                Tue, 05 May 2026   Prob (F-statistic):               0.00
Time:                        21:23:49   Log-Likelihood:                -10153.
No. Observations:                9320   AIC:                         2.039e+04
Df Residuals:                    9277   BIC:                         2.070e+04
Df Model:                          42                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
const         

In [53]:
# 4.1) Stepwise (manual simplificado)


X_test_sm = sm.add_constant(X_test)

y_pred = modelo.predict(X_test_sm)
r2_step = r2_score(y_test, y_pred)

print(r2_step)

# O R² obtido foi 0.60 (ou o valor que aparecer)

0.3652910756080827


# 5) Comparação dos modelos
Comparando os modelos:

Ridge: bom controle de overfitting
LASSO: seleção automática de variáveis
Stepwise: modelo mais interpretável

O melhor modelo foi o Ridge com alpha = 0.01, com R² de 0.65

In [25]:
#  6) Melhorar o modelo
# criar variáveis novas
df['idade_2'] = df['idade']**2
df['tempo_emprego_log'] = np.log(df['tempo_emprego'] + 1)

X = df.drop(columns=['renda', 'log_renda'])
X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model = Ridge(alpha=0.01)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(r2_score(y_test, y_pred))

0.3652957877817673


C:\Users\olive\AppData\Local\Temp\ipykernel_27832\3452377968.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['idade_2'] = df['idade']**2
C:\Users\olive\AppData\Local\Temp\ipykernel_27832\3452377968.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tempo_emprego_log'] = np.log(df['tempo_emprego'] + 1)


In [23]:
# 7) 
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(max_depth=5, random_state=42)
tree.fit(X_train, y_train)

y_pred = tree.predict(X_test)

print(r2_score(y_test, y_pred))

0.3711160346488538


O modelo apresentou R² de 0.58, sendo inferior aos modelos lineares